In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'applied jobs',
 'Rataj_Adel_El-Sayed_Hassan__Resume  (1).pdf',
 'Rataj_Adel_El-Sayed_Hassan__Resume .pdf',
 'Rataj Adel',
 'BBC News Summary',
 'my CV',
 'Rataj_Adel_El-Sayed_Hassan_Resume_ (1).pdf',
 'Rataj Adel El Sayed Hassan.pdf',
 'AI_Lab_Sheet.pdf',
 'supply_chain_system']

In [4]:
os.listdir('/content/drive/MyDrive/supply_chain_system')

['DataCoSupplyChainDataset_Cleaned.csv',
 'DescriptionDataCoSupplyChain_Cleaned.csv',
 'tokenized_access_logs_Cleaned.csv',
 'Data preprocessing (2).py',
 'data after clean(DataCoSupplyChainDatase).rar',
 'model_full (2).csv']

In [5]:
DATACO_PATH = "DataCoSupplyChain_Cleaned.csv"
LOGS_PATH   = "tokenized_access_logs_Cleaned.csv"
MODEL_PATH  = "model_full.csv"


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
DATACO_PATH = "/content/drive/MyDrive/supply_chain_system/DataCoSupplyChainDataset_Cleaned.csv"

LOGS_PATH = "/content/drive/MyDrive/supply_chain_system/tokenized_access_logs_Cleaned.csv"

MODEL_PATH = "/content/drive/MyDrive/supply_chain_system/model_full (2).csv"

In [20]:
import os

# لو الملفات في Colab مباشرة
print("📁 ملفات في /content:")
for f in os.listdir("/content"):
    if f.endswith(".csv"):
        print(f"   ✅ {f}")

# لو الملفات على Drive
print("\n📁 ملفات في Drive (supply_chain):")
drive_path = "/content/drive/MyDrive/supply_chain_system"
if os.path.exists(drive_path):
    for f in os.listdir(drive_path):
        if f.endswith(".csv"):
            print(f"   ✅ {f}")
else:
    print("   ❌ الفولدر مش موجود — اعمليه على Drive الأول")

📁 ملفات في /content:
   ✅ DataCoSupplyChainDataset_Cleaned.csv
   ✅ tokenized_access_logs_Cleaned.csv
   ✅ model_full (2).csv

📁 ملفات في Drive (supply_chain):
   ✅ DataCoSupplyChainDataset_Cleaned.csv
   ✅ DescriptionDataCoSupplyChain_Cleaned.csv
   ✅ tokenized_access_logs_Cleaned.csv
   ✅ model_full (2).csv


In [11]:
from google.colab import files

print("اختاري الملفات التلاتة:")
uploaded = files.upload()

# هيظهرلك أسماء الملفات اللي اترفعت
print("\n✅ الملفات اللي اترفعت:")
for name in uploaded.keys():
    print(f"   - {name}")

اختاري الملفات التلاتة:


Saving model_full (2).csv to model_full (2).csv

✅ الملفات اللي اترفعت:
   - model_full (2).csv


In [12]:
from google.colab import files

print("اختاري الملفات التلاتة:")
uploaded = files.upload()

# هيظهرلك أسماء الملفات اللي اترفعت
print("\n✅ الملفات اللي اترفعت:")
for name in uploaded.keys():
    print(f"   - {name}")

اختاري الملفات التلاتة:


Saving DataCoSupplyChainDataset_Cleaned.csv to DataCoSupplyChainDataset_Cleaned.csv

✅ الملفات اللي اترفعت:
   - DataCoSupplyChainDataset_Cleaned.csv


In [13]:
from google.colab import files

print("اختاري الملفات التلاتة:")
uploaded = files.upload()

# هيظهرلك أسماء الملفات اللي اترفعت
print("\n✅ الملفات اللي اترفعت:")
for name in uploaded.keys():
    print(f"   - {name}")

اختاري الملفات التلاتة:


Saving tokenized_access_logs_Cleaned.csv to tokenized_access_logs_Cleaned.csv

✅ الملفات اللي اترفعت:
   - tokenized_access_logs_Cleaned.csv


In [26]:
import os

base_path = "/content/drive/MyDrive/supply_chain_system"

files = {
    "supply_chain": os.path.join(
        base_path, "DataCoSupplyChainDataset_Cleaned.csv"
    ),
    "access_logs": os.path.join(
        base_path, "tokenized_access_logs_Cleaned.csv"
    ),
    "model": os.path.join(
        base_path, "model_full (2).csv"
    )
}

for name, path in files.items():
    if os.path.exists(path):
        print(f"✅ موجود: {name}")
    else:
        print(f"❌ مش موجود: {path}")

✅ موجود: supply_chain
✅ موجود: access_logs
✅ موجود: model


In [29]:
dataco = pd.read_csv(DATACO_PATH)
logs   = pd.read_csv(LOGS_PATH)
model  = pd.read_csv(MODEL_PATH)

print(f"   DataCo:      {dataco.shape[0]:,} صف × {dataco.shape[1]} عمود")
print(f"   Access Logs: {logs.shape[0]:,} صف × {logs.shape[1]} عمود")
print(f"   model_full:  {model.shape[0]:,} صف × {model.shape[1]} عمود")

   DataCo:      180,519 صف × 65 عمود
   Access Logs: 463,176 صف × 7 عمود
   model_full:  83,750 صف × 54 عمود


In [30]:
# ============================================================
# 4) تجهيز DataCo (الأوردرات)
# ============================================================
print("\n🔧 بنلخّص DataCo...")

dataco["order_datetime"] = pd.to_datetime(
    dataco["order date (DateOrders)"], errors="coerce"
)
dataco["year"]  = dataco["order_datetime"].dt.year
dataco["month"] = dataco["order_datetime"].dt.month

dataco_features = (
    dataco
    .groupby(["Category Name", "year", "month"])
    .agg(
        dc_total_sales      = ("Sales", "sum"),
        dc_total_orders     = ("Order Id", "nunique"),
        dc_avg_discount     = ("Order Item Discount Rate", "mean"),
        dc_late_rate        = ("Late_delivery_risk", "mean"),
        dc_avg_profit_ratio = ("Order Item Profit Ratio", "mean"),
        dc_total_benefit    = ("Benefit per order", "sum"),
    )
    .reset_index()
    .rename(columns={"Category Name": "category"})
)
print(f"   ✅ DataCo features: {dataco_features.shape[0]:,} صف")


🔧 بنلخّص DataCo...
   ✅ DataCo features: 853 صف


In [31]:
# ============================================================
# 5) تجهيز Access Logs (الزيارات)
# ============================================================
print("\n🔧 بنلخّص Access Logs...")

logs["log_datetime"] = pd.to_datetime(logs["log_datetime"], errors="coerce")
logs["year"]  = logs["log_datetime"].dt.year
logs["month"] = logs["log_datetime"].dt.month

logs_features = (
    logs
    .groupby(["Department", "year", "month"])
    .agg(
        lg_total_views     = ("ip", "count"),
        lg_unique_visitors = ("ip", "nunique"),
        lg_unique_products = ("Product", "nunique"),
    )
    .reset_index()
    .rename(columns={"Department": "category"})
)
print(f"   ✅ Access Logs features: {logs_features.shape[0]:,} صف")


🔧 بنلخّص Access Logs...
   ✅ Access Logs features: 30 صف


In [32]:
print("\n🔧 بنلخّص model_full...")

model["Date"] = pd.to_datetime(model["Date"], errors="coerce")
model["year"]  = model["Date"].dt.year
model["month"] = model["Date"].dt.month

model_sku_features = (
    model
    .groupby(["SKU_Code", "year", "month"])
    .agg(
        mfg_units_sold_total = ("Units_Sold", "sum"),
        mfg_avg_inventory    = ("Inventory_Level", "mean"),
        mfg_below_rop_rate   = ("Below_ROP", "mean"),
        mfg_avg_lead_time    = ("Supplier_Lead_Time_Days", "mean"),
        mfg_avg_days_cover   = ("Days_of_Cover", "mean"),
        mfg_promo_rate       = ("Promotion_Flag", "mean"),
        mfg_avg_forecast     = ("Demand_Forecast", "mean"),
    )
    .reset_index()
)
print(f"   ✅ model_full SKU features: {model_sku_features.shape[0]:,} صف")


🔧 بنلخّص model_full...
   ✅ model_full SKU features: 600 صف


In [33]:
# ============================================================
# 7) جدول ربط SKU → Category
#    ⚠️ ده مثال — عدّليه حسب الفئات الحقيقية عندك
# ============================================================
print("\n🔗 بنعمل جدول ربط SKU → Category...")

# خدي كل SKU موجود في الداتا
unique_skus = model["SKU_Code"].unique()

# ⚠️ هنا بنحط "Unknown" مؤقتًا — لازم تحدّدي كل SKU بيتبع أنهي فئة
#    لو مش عارفة، سيبيه Unknown وشوفي النتيجة
sku_to_category = pd.DataFrame({
    "SKU_Code": unique_skus,
    "category": "Unknown"
})



🔗 بنعمل جدول ربط SKU → Category...


In [35]:
import pandas as pd

# قراءة الملف مباشرة باستخدام المسار المعرف في الدكشنري
sku_to_category = pd.read_csv(files["model"]) # أو المتغير الخاص بملف الـ CSV بتاع الـ SKU

# عرض أول عينة للتأكد إن البيانات اتحملت صح
print(sku_to_category.head())

# تطبيق الـ Merge بتاعك عادي جداً
model_sku_features = model_sku_features.merge(
    sku_to_category, on="SKU_Code", how="left"
)

         Date  Units_Sold  Inventory_Level  Supplier_Lead_Time_Days  \
0  2024-01-31          18         2.244453                 1.539443   
1  2024-02-01          34         1.981888                 1.539443   
2  2024-02-02          24         1.796548                 1.539443   
3  2024-02-03          22         1.626653                 1.539443   
4  2024-02-04          32         1.379534                 1.539443   

   Reorder_Point  Unit_Cost  Unit_Price  Promotion_Flag  Demand_Forecast  \
0       1.438275   0.381792    0.311497               0        -0.824247   
1       1.438275   0.381792    0.311497               0         1.163020   
2       1.438275   0.381792    0.311497               0         0.273023   
3       1.438275   0.381792    0.311497               0        -0.116951   
4       1.438275   0.381792    0.311497               1         0.462134   

   WeekOfYear  ...  Supplier_ID_SUP_10  Supplier_ID_SUP_2  Supplier_ID_SUP_3  \
0   -1.708754  ...                  

In [37]:
print(model_sku_features.columns.tolist())

['SKU_Code', 'year', 'month', 'mfg_units_sold_total', 'mfg_avg_inventory', 'mfg_below_rop_rate', 'mfg_avg_lead_time', 'mfg_avg_days_cover', 'mfg_promo_rate', 'mfg_avg_forecast', 'Date', 'Units_Sold', 'Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag', 'Demand_Forecast', 'WeekOfYear', 'IsWeekend', 'IsMonthEnd', 'Month_sin', 'Month_cos', 'DOW_sin', 'DOW_cos', 'Unit_Margin', 'Margin_Pct', 'Inventory_vs_ROP', 'Below_ROP', 'Order_Placed', 'Order_Quantity_log', 'Sold_lag1', 'Sold_lag7', 'Sold_lag14', 'Sold_roll7_mean', 'Sold_roll14_mean', 'Sold_roll30_mean', 'Sold_roll7_std', 'Forecast_lag1_err', 'Days_of_Cover', 'Inventory_Level_Outlier', 'Demand_Forecast_Outlier', 'Warehouse_ID_WH_1', 'Warehouse_ID_WH_2', 'Warehouse_ID_WH_3', 'Warehouse_ID_WH_4', 'Warehouse_ID_WH_5', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'Supplier_ID_SUP_1', 'Supplier_ID_SUP_10', 'Supplier_ID_SUP_2', 'Supplier_ID_SUP_3', 'Supplier_ID_SUP_4', 'S

In [38]:
# شوفي الأعمدة اللي جوه ملف الـ mapping عشان تعرفي الاسم الصح للـ category إيه
print("أعمدة ملف الـ Mapping:", sku_to_category.columns.tolist())

أعمدة ملف الـ Mapping: ['Date', 'Units_Sold', 'Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag', 'Demand_Forecast', 'WeekOfYear', 'IsWeekend', 'IsMonthEnd', 'Month_sin', 'Month_cos', 'DOW_sin', 'DOW_cos', 'Unit_Margin', 'Margin_Pct', 'Inventory_vs_ROP', 'Below_ROP', 'Order_Placed', 'Order_Quantity_log', 'Sold_lag1', 'Sold_lag7', 'Sold_lag14', 'Sold_roll7_mean', 'Sold_roll14_mean', 'Sold_roll30_mean', 'Sold_roll7_std', 'Forecast_lag1_err', 'Days_of_Cover', 'Inventory_Level_Outlier', 'Demand_Forecast_Outlier', 'SKU_Code', 'Warehouse_ID_WH_1', 'Warehouse_ID_WH_2', 'Warehouse_ID_WH_3', 'Warehouse_ID_WH_4', 'Warehouse_ID_WH_5', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'Supplier_ID_SUP_1', 'Supplier_ID_SUP_10', 'Supplier_ID_SUP_2', 'Supplier_ID_SUP_3', 'Supplier_ID_SUP_4', 'Supplier_ID_SUP_5', 'Supplier_ID_SUP_6', 'Supplier_ID_SUP_7', 'Supplier_ID_SUP_8', 'Supplier_ID_SUP_9', 'Split']


In [36]:
# نجمّع على مستوى الفئة + الشهر
model_features = (
    model_sku_features
    .groupby(["category", "year", "month"])
    .agg(
        mfg_units_sold_total = ("mfg_units_sold_total", "sum"),
        mfg_avg_inventory    = ("mfg_avg_inventory", "mean"),
        mfg_below_rop_rate   = ("mfg_below_rop_rate", "mean"),
        mfg_avg_lead_time    = ("mfg_avg_lead_time", "mean"),
        mfg_avg_days_cover   = ("mfg_avg_days_cover", "mean"),
        mfg_promo_rate       = ("mfg_promo_rate", "mean"),
        mfg_avg_forecast     = ("mfg_avg_forecast", "mean"),
    )
    .reset_index()
)
print(f"   ✅ model_full features (بعد الربط): {model_features.shape[0]:,} صف")


KeyError: 'category'

In [39]:
# ============================================================
# الخطوة 2: model_full → demand_shortage_features.csv
# المستوى: SKU_Code × year × month
# ============================================================
print("="*60)
print("📦 بنعالج model_full (المخزون والطلب)...")
print("="*60)

model = pd.read_csv(MODEL_PATH)
print(f"   الحجم الأصلي: {model.shape[0]:,} صف × {model.shape[1]} عمود")

# --- تحويل التاريخ ---
model["Date"] = pd.to_datetime(model["Date"], errors="coerce")
model["year"]  = model["Date"].dt.year
model["month"] = model["Date"].dt.month

# --- التجميع على SKU + سنة + شهر ---
demand_features = (
    model
    .groupby(["SKU_Code", "year", "month"])
    .agg(
        mfg_units_sold_total = ("Units_Sold", "sum"),                # إجمالي الوحدات المبيعة
        mfg_units_sold_mean  = ("Units_Sold", "mean"),               # متوسط الوحدات المبيعة
        mfg_avg_inventory    = ("Inventory_Level", "mean"),          # متوسط المخزون (مُعاير)
        mfg_below_rop_rate   = ("Below_ROP", "mean"),                # نسبة الأيام تحت ROP
        mfg_avg_lead_time    = ("Supplier_Lead_Time_Days", "mean"),  # متوسط مدة التوريد
        mfg_avg_days_cover   = ("Days_of_Cover", "mean"),            # متوسط أيام التغطية
        mfg_promo_rate       = ("Promotion_Flag", "mean"),           # نسبة أيام العروض
        mfg_avg_forecast     = ("Demand_Forecast", "mean"),          # متوسط توقع الطلب
        mfg_avg_inv_vs_rop   = ("Inventory_vs_ROP", "mean"),         # متوسط الفرق بين المخزون و ROP
    )
    .reset_index()
)

# --- Features مشتقة ---
# 1) خطر نقص مخزون: لو أكتر من 50% من الأيام تحت ROP
demand_features["stockout_risk_flag"] = (
    demand_features["mfg_below_rop_rate"] > 0.5
).astype(int)

# 2) خطر نفاد وشيك: أيام التغطية أقل من Lead Time
demand_features["urgent_shortage_flag"] = (
    demand_features["mfg_avg_days_cover"] < demand_features["mfg_avg_lead_time"]
).astype(int)

# 3) عمود year_month للعرض
demand_features["year_month"] = (
    demand_features["year"].astype(int).astype(str) + "-" +
    demand_features["month"].astype(int).astype(str).str.zfill(2)
)

# --- الحفظ ---
demand_features.to_csv("demand_shortage_features.csv", index=False, encoding="utf-8-sig")

print(f"   ✅ الحجم النهائي: {demand_features.shape[0]:,} صف × {demand_features.shape[1]} عمود")
print(f"   ✅ اتحفظ: demand_shortage_features.csv")
print("\n📊 أول 5 صفوف:")
display(demand_features.head())

# --- إحصائيات سريعة ---
print(f"\n📈 إحصائيات:")
print(f"   عدد SKUs:            {demand_features['SKU_Code'].nunique()}")
print(f"   عدد الشهور:          {demand_features['year_month'].nunique()}")
print(f"   SKUs فيها خطر نقص:   {demand_features['stockout_risk_flag'].sum()}")
print(f"   SKUs فيها خطر نفاد:  {demand_features['urgent_shortage_flag'].sum()}")

📦 بنعالج model_full (المخزون والطلب)...
   الحجم الأصلي: 83,750 صف × 54 عمود
   ✅ الحجم النهائي: 600 صف × 15 عمود
   ✅ اتحفظ: demand_shortage_features.csv

📊 أول 5 صفوف:


,SKU_Code,year,month,mfg_units_sold_total,mfg_units_sold_mean,mfg_avg_inventory,mfg_below_rop_rate,mfg_avg_lead_time,mfg_avg_days_cover,mfg_promo_rate,mfg_avg_forecast,mfg_avg_inv_vs_rop,stockout_risk_flag,urgent_shortage_flag,year_month
0,1,2024,1,109,21.800000,0.452835,0.000000,0.055273,-0.109218,0.000000,-0.105412,0.135651,0,1,2024-01
1,1,2024,2,3887,26.806897,0.314576,0.082759,0.055273,-0.307660,0.068966,0.441801,-0.016256,0,1,2024-02
2,1,2024,3,4618,29.793548,0.121814,0.103226,0.055273,-0.548609,0.129032,0.799404,-0.229578,0,1,2024-03
3,1,2024,4,4579,30.526667,0.382457,0.073333,0.055273,-0.512861,0.086667,0.840232,0.058305,0,1,2024-04
4,1,2024,5,4231,27.296774,0.306606,0.070968,0.055273,-0.395969,0.077419,0.489879,-0.026104,0,1,2024-05



📈 إحصائيات:
   عدد SKUs:            50
   عدد الشهور:          12
   SKUs فيها خطر نقص:   0
   SKUs فيها خطر نفاد:  276


In [40]:
# ============================================================
# الخطوة 3: DataCo + Logs → market_signals.csv
# المستوى: category × year × month
# ============================================================
print("="*60)
print("🛒 بنعالج DataCo + Access Logs (إشارات السوق)...")
print("="*60)

# ---------- 3.1: DataCo ----------
print("\n🔧 DataCo (الأوردرات)...")
dataco = pd.read_csv(DATACO_PATH)
print(f"   الحجم الأصلي: {dataco.shape[0]:,} صف × {dataco.shape[1]} عمود")

# تحويل التاريخ
dataco["order_datetime"] = pd.to_datetime(
    dataco["order date (DateOrders)"], errors="coerce"
)
dataco["year"]  = dataco["order_datetime"].dt.year
dataco["month"] = dataco["order_datetime"].dt.month

# التجميع على الفئة + سنة + شهر
dataco_features = (
    dataco
    .groupby(["Category Name", "year", "month"])
    .agg(
        dc_total_sales      = ("Sales", "sum"),                     # إجمالي المبيعات
        dc_total_orders     = ("Order Id", "nunique"),              # عدد الأوردرات الفريدة
        dc_avg_discount     = ("Order Item Discount Rate", "mean"), # متوسط نسبة الخصم
        dc_late_rate        = ("Late_delivery_risk", "mean"),       # نسبة التأخير
        dc_avg_profit_ratio = ("Order Item Profit Ratio", "mean"),  # متوسط نسبة الربح
        dc_total_benefit    = ("Benefit per order", "sum"),         # إجمالي الربح
    )
    .reset_index()
    .rename(columns={"Category Name": "category"})
)
print(f"   ✅ DataCo features: {dataco_features.shape[0]:,} صف")


# ---------- 3.2: Access Logs ----------
print("\n🔧 Access Logs (الزيارات)...")
logs = pd.read_csv(LOGS_PATH)
print(f"   الحجم الأصلي: {logs.shape[0]:,} صف × {logs.shape[1]} عمود")

# تحويل التاريخ
logs["log_datetime"] = pd.to_datetime(logs["log_datetime"], errors="coerce")
logs["year"]  = logs["log_datetime"].dt.year
logs["month"] = logs["log_datetime"].dt.month

# التجميع على القسم + سنة + شهر
logs_features = (
    logs
    .groupby(["Department", "year", "month"])
    .agg(
        lg_total_views     = ("ip", "count"),           # إجمالي الزيارات
        lg_unique_visitors = ("ip", "nunique"),         # عدد الزوار الفريدين
        lg_unique_products = ("Product", "nunique"),    # عدد المنتجات المختلفة
    )
    .reset_index()
    .rename(columns={"Department": "category"})
)
print(f"   ✅ Logs features: {logs_features.shape[0]:,} صف")


# ---------- 3.3: الدمج (outer join على category + year + month) ----------
print("\n🔀 بندمج DataCo + Logs...")
market_signals = pd.merge(
    dataco_features,
    logs_features,
    on=["category", "year", "month"],
    how="outer"   # outer عشان نحتفظ بكل الفئات حتى لو مش موجودة في الاتنين
)

# نملأ القيم الناقصة في الأعمدة الرقمية بـ 0
numeric_cols = market_signals.select_dtypes(include=[np.number]).columns
market_signals[numeric_cols] = market_signals[numeric_cols].fillna(0)

# ---------- 3.4: Features مشتقة ----------
# 1) نسبة التحويل: أوردرات ÷ زيارات
market_signals["conversion_rate"] = np.where(
    market_signals["lg_total_views"] > 0,
    market_signals["dc_total_orders"] / market_signals["lg_total_views"],
    0
)

# 2) اهتمام عالي بس مبيعات قليلة
views_mean = market_signals["lg_total_views"].mean()
sales_mean = market_signals["dc_total_sales"].mean()
market_signals["high_interest_low_sales"] = (
    (market_signals["lg_total_views"] > views_mean) &
    (market_signals["dc_total_sales"] < sales_mean)
).astype(int)

# 3) تأخير عالي
market_signals["high_late_risk"] = (
    market_signals["dc_late_rate"] > 0.3
).astype(int)

# 4) عمود year_month
market_signals["year_month"] = (
    market_signals["year"].astype(int).astype(str) + "-" +
    market_signals["month"].astype(int).astype(str).str.zfill(2)
)

# ---------- 3.5: الحفظ ----------
market_signals.to_csv("market_signals.csv", index=False, encoding="utf-8-sig")

print(f"\n   ✅ الحجم النهائي: {market_signals.shape[0]:,} صف × {market_signals.shape[1]} عمود")
print(f"   ✅ اتحفظ: market_signals.csv")
print("\n📊 أول 5 صفوف:")
display(market_signals.head())

🛒 بنعالج DataCo + Access Logs (إشارات السوق)...

🔧 DataCo (الأوردرات)...
   الحجم الأصلي: 180,519 صف × 65 عمود
   ✅ DataCo features: 853 صف

🔧 Access Logs (الزيارات)...
   الحجم الأصلي: 463,176 صف × 7 عمود
   ✅ Logs features: 30 صف

🔀 بندمج DataCo + Logs...

   ✅ الحجم النهائي: 883 صف × 16 عمود
   ✅ اتحفظ: market_signals.csv

📊 أول 5 صفوف:


,category,year,month,dc_total_sales,dc_total_orders,dc_avg_discount,dc_late_rate,dc_avg_profit_ratio,dc_total_benefit,lg_total_views,lg_unique_visitors,lg_unique_products,conversion_rate,high_interest_low_sales,high_late_risk,year_month
0,Accessories,2015,1,3848.459970,56.0,0.110536,0.571429,0.128929,480.579995,0.0,0.0,0.0,0.0,0,1,2015-01
1,Accessories,2015,2,3723.509969,51.0,0.106538,0.596154,0.194231,622.910004,0.0,0.0,0.0,0.0,0,1,2015-02
2,Accessories,2015,3,4523.189989,65.0,0.104848,0.606061,0.187879,804.890012,0.0,0.0,0.0,0.0,0,1,2015-03
3,Accessories,2015,4,3848.459966,53.0,0.110755,0.509434,-0.007358,-61.439993,0.0,0.0,0.0,0.0,0,1,2015-04
4,Accessories,2015,5,5822.669919,70.0,0.097571,0.628571,0.177857,1091.880002,0.0,0.0,0.0,0.0,0,1,2015-05


In [41]:
# ============================================================
# الخطوة 4: تقرير جودة وتطابق الفئات
# ============================================================
print("="*60)
print("📋 تقرير جودة الدمج")
print("="*60)

# ---------- 4.1: تطابق الفئات ----------
dataco_cats = set(dataco_features["category"].dropna().unique())
logs_cats   = set(logs_features["category"].dropna().unique())
overlap     = dataco_cats & logs_cats

print(f"\n🔍 تطابق الفئات بين DataCo و Logs:")
print(f"   فئات DataCo:        {len(dataco_cats)}")
print(f"   فئات Logs:          {len(logs_cats)}")
print(f"   متطابقة (overlap):  {len(overlap)}")

if overlap:
    print(f"\n   ✅ الفئات المتطابقة:")
    for c in sorted(overlap):
        print(f"      - {c}")

if dataco_cats - logs_cats:
    print(f"\n   ⚠️ موجودة في DataCo بس:")
    for c in sorted(dataco_cats - logs_cats):
        print(f"      - {c}")

if logs_cats - dataco_cats:
    print(f"\n   ⚠️ موجودة في Logs بس:")
    for c in sorted(logs_cats - dataco_cats):
        print(f"      - {c}")

# ---------- 4.2: جودة market_signals ----------
print(f"\n📊 جودة market_signals:")
total = len(market_signals)
has_dc = (market_signals["dc_total_sales"] != 0).sum()
has_lg = (market_signals["lg_total_views"] != 0).sum()
both   = ((market_signals["dc_total_sales"] != 0) &
          (market_signals["lg_total_views"] != 0)).sum()

print(f"   إجمالي الصفوف:              {total:,}")
print(f"   فيها بيانات DataCo:         {has_dc:,} ({has_dc/total:.1%})")
print(f"   فيها بيانات Logs:           {has_lg:,} ({has_lg/total:.1%})")
print(f"   فيها الاتنين مع بعض:        {both:,} ({both/total:.1%})")

# ---------- 4.3: جودة demand_features ----------
print(f"\n📊 جودة demand_shortage_features:")
total2 = len(demand_features)
print(f"   إجمالي الصفوف:              {total2:,}")
print(f"   عدد SKUs:                   {demand_features['SKU_Code'].nunique()}")
print(f"   عدد الشهور:                 {demand_features['year_month'].nunique()}")
print(f"   SKUs فيها خطر نقص:          {demand_features['stockout_risk_flag'].sum()}")
print(f"   SKUs فيها خطر نفاد وشيك:    {demand_features['urgent_shortage_flag'].sum()}")

print("\n" + "="*60)
print("✅ تم!")
print("="*60)

📋 تقرير جودة الدمج

🔍 تطابق الفئات بين DataCo و Logs:
   فئات DataCo:        50
   فئات Logs:          6
   متطابقة (overlap):  0

   ⚠️ موجودة في DataCo بس:
      - Accessories
      - As Seen on  TV!
      - Baby 
      - Baseball & Softball
      - Basketball
      - Books 
      - Boxing & MMA
      - CDs 
      - Cameras 
      - Camping & Hiking
      - Cardio Equipment
      - Children's Clothing
      - Cleats
      - Computers
      - Consumer Electronics
      - Crafts
      - DVDs
      - Electronics
      - Fishing
      - Fitness Accessories
      - Garden
      - Girls' Apparel
      - Golf Apparel
      - Golf Bags & Carts
      - Golf Balls
      - Golf Gloves
      - Golf Shoes
      - Health and Beauty
      - Hockey
      - Hunting & Shooting
      - Indoor/Outdoor Games
      - Kids' Golf Clubs
      - Lacrosse
      - Men's Clothing
      - Men's Footwear
      - Men's Golf Clubs
      - Music
      - Pet Supplies
      - Shop By Sport
      - Soccer
      - Sporti

In [42]:
# ============================================================
# الخطوة 5: تنزيل الملفات الناتجة
# ============================================================
from google.colab import files

print("⬇️ بننزّل الملفات...")

# ننزّل كل ملف لوحده (أضمن في Colab)
for f in ["demand_shortage_features.csv", "market_signals.csv"]:
    if os.path.exists(f):
        print(f"   ⬇️ {f}")
        files.download(f)
    else:
        print(f"   ❌ {f} مش موجود")

print("\n🎉 خلصنا!")

⬇️ بننزّل الملفات...
   ⬇️ demand_shortage_features.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ⬇️ market_signals.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 خلصنا!
